# 06 — Convert KQA Pro CSV to JSONL with Manual Russian Translation Fields

This notebook converts the manually selected KQA Pro CSV into JSONL and prepares fields for manual Russian translation.

There is **no LLM/API translation** in this version. The workflow is intentionally simple:

1. Read `kqapro_manual_review_sheet.csv`.
2. Keep rows selected for reconstruction.
3. Export a JSONL file with empty Russian translation fields.
4. Export a convenient CSV review sheet where translations can be pasted manually.
5. Optionally rebuild the final JSONL after the review CSV has been filled.

The notebook is append-safe: existing output files are not overwritten unless `OVERWRITE_OUTPUT = True`.

## Table of contents

- [1. Configuration](#1-configuration)
- [2. Imports and helpers](#2-imports-and-helpers)
- [3. Load and validate source CSV](#3-load-and-validate-source-csv)
- [4. Select and normalize records](#4-select-and-normalize-records)
- [5. Build manual translation review table](#5-build-manual-translation-review-table)
- [6. Export JSONL with manual translation slots](#6-export-jsonl-with-manual-translation-slots)
- [7. Optional: rebuild final JSONL from edited review CSV](#7-optional-rebuild-final-jsonl-from-edited-review-csv)
- [8. Quick checks](#8-quick-checks)
- [9. Notes and conclusions](#9-notes-and-conclusions)

## 1. Configuration

In [37]:
from pathlib import Path

# Source CSV produced by the previous KQA Pro manual review/filtering step.
INPUT_CSV = Path("kqapro_manual_review_sheet.csv")

# Output directory for converted artifacts.
OUTPUT_DIR = Path("artifacts_kqapro_stage6")
CSV_DIR = OUTPUT_DIR / "csv"
JSONL_DIR = OUTPUT_DIR / "jsonl"
REPORTS_DIR = OUTPUT_DIR / "reports"

# Main outputs.
REVIEW_CSV_OUT = CSV_DIR / "kqapro_manual_translation_review_sheet.csv"
JSONL_DRAFT_OUT = JSONL_DIR / "kqapro_selected_manual_translation_draft.jsonl"
JSONL_FINAL_OUT = JSONL_DIR / "kqapro_selected_ru_final.jsonl"
REPORT_OUT = REPORTS_DIR / "stage6_manual_translation_conversion_report.md"

# Safety switch. Keep False to avoid overwriting previous artifacts.
OVERWRITE_OUTPUT = False

# Source CSV columns.
ID_COL = "benchmark_id"
QUESTION_EN_COL = "question_en_original"
ANSWER_TEXT_COL = "answer_text"

# Selection rule. Only rows with this recommended_action are exported.
SELECTED_ACTION = "keep_for_reconstruction"

# Manual translation columns created by this notebook.
QUESTION_RU_MANUAL_COL = "question_ru_manual"
TRANSLATION_COMMENT_COL = "translation_comment"

## 2. Imports and helpers

In [38]:
import csv
import json
import re
from datetime import datetime
from typing import Any, Dict, Iterable, List, Optional

import pandas as pd

for d in [CSV_DIR, JSONL_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def safe_write_path(path: Path, overwrite: bool = False) -> Path:
    """Return a writable path without deleting or overwriting existing artifacts."""
    if overwrite or not path.exists():
        return path
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return path.with_name(f"{path.stem}_{stamp}{path.suffix}")


def clean_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    text = str(value)
    return re.sub(r"\s+", " ", text).strip()


def parse_answer_text(answer_text: str) -> List[str]:
    """Parse comma-separated answers conservatively; keep the raw answer text too."""
    answer_text = clean_text(answer_text)
    if not answer_text:
        return []
    return [part.strip() for part in answer_text.split(",") if part.strip()]


def write_jsonl(path: Path, rows: Iterable[Dict[str, Any]], overwrite: bool = False) -> Path:
    out_path = safe_write_path(path, overwrite=overwrite)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    return out_path


def write_csv(path: Path, df: pd.DataFrame, overwrite: bool = False) -> Path:
    out_path = safe_write_path(path, overwrite=overwrite)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False, encoding="utf-8")
    return out_path


def review_csv_candidates() -> List[Path]:
    """Return existing review CSV files, newest first, including timestamped copies."""
    candidates = list(CSV_DIR.glob(f"{REVIEW_CSV_OUT.stem}*.csv"))
    candidates = [p for p in candidates if p.exists()]
    return sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)


def count_filled_translations(path: Path) -> int:
    try:
        tmp = pd.read_csv(path, dtype=str, keep_default_na=False)
    except Exception:
        return 0
    if QUESTION_RU_MANUAL_COL not in tmp.columns:
        return 0
    return int(tmp[QUESTION_RU_MANUAL_COL].map(clean_text).str.len().gt(0).sum())


def find_best_review_csv() -> Optional[Path]:
    """Pick the review CSV with the largest number of filled manual translations."""
    candidates = review_csv_candidates()
    if not candidates:
        return None
    return max(candidates, key=lambda p: (count_filled_translations(p), p.stat().st_mtime))


def load_existing_manual_translations() -> Dict[str, Dict[str, str]]:
    """Load already typed translations so rebuilding the review table does not lose them."""
    best_path = find_best_review_csv()
    if best_path is None:
        return {}

    existing_df = pd.read_csv(best_path, dtype=str, keep_default_na=False)
    if ID_COL not in existing_df.columns:
        return {}

    result = {}
    for _, row in existing_df.iterrows():
        benchmark_id = clean_text(row.get(ID_COL, ""))
        if not benchmark_id:
            continue
        result[benchmark_id] = {
            QUESTION_RU_MANUAL_COL: clean_text(row.get(QUESTION_RU_MANUAL_COL, "")),
            TRANSLATION_COMMENT_COL: clean_text(row.get(TRANSLATION_COMMENT_COL, "")),
        }
    print(f"Loaded existing manual translations from: {best_path}")
    print(f"Filled translations found: {count_filled_translations(best_path)}")
    return result


## 3. Load and validate source CSV

In [39]:
assert INPUT_CSV.exists(), f"Input CSV not found: {INPUT_CSV.resolve()}"

# keep_default_na=False prevents labels like "NA" from becoming NaN.
df = pd.read_csv(
    INPUT_CSV,
    dtype=str,
    keep_default_na=False,
    quoting=csv.QUOTE_MINIMAL,
)

required_cols = [ID_COL, QUESTION_EN_COL, ANSWER_TEXT_COL]
missing_cols = [col for col in required_cols if col not in df.columns]
assert not missing_cols, f"Missing required columns: {missing_cols}"

print(f"Loaded rows: {len(df)}")
print(f"Loaded columns: {list(df.columns)}")
df.head(3)

Loaded rows: 74
Loaded columns: ['benchmark_id', 'primary_reasoning_family', 'recommended_action', 'reconstructability_score_v2', 'program_signature', 'program_length', 'hop_count_estimate', 'question_en_original', 'answer_text', 'manual_decision', 'manual_bucket', 'russian_reconstruction_idea', 'target_domain_ru', 'notes']


,benchmark_id,primary_reasoning_family,recommended_action,reconstructability_score_v2,program_signature,program_length,hop_count_estimate,question_en_original,answer_text,manual_decision,manual_bucket,russian_reconstruction_idea,target_domain_ru,notes
0,kqapro_train_013987,multi_hop,keep_for_reconstruction,26,Find -> Relate -> QFilterYear -> FilterConcept...,12,3,"Which big city was, in 1958, a twinned adminis...",Marseille,,,,,
1,kqapro_train_035873,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> Fil...,12,3,"Who is a cast member of ""Taken"" (which also ha...",Michael Jeter,,,,,
2,kqapro_train_075279,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> Fil...,10,3,"What person is a member, starting in 2012, of ...",Thiago Silva,,,,,


## 4. Select and normalize records

In [40]:
# Keep only rows selected for reconstruction if the review column exists.
if "recommended_action" in df.columns:
    selected_df = df[df["recommended_action"].map(clean_text).eq(SELECTED_ACTION)].copy()
else:
    selected_df = df.copy()

selected_df[QUESTION_EN_COL] = selected_df[QUESTION_EN_COL].map(clean_text)
selected_df = selected_df[selected_df[QUESTION_EN_COL].str.len() > 0].copy()

before = len(selected_df)
selected_df = selected_df.drop_duplicates(subset=[ID_COL], keep="first").copy()
print(f"Selected rows: {len(selected_df)}")
print(f"Removed duplicate benchmark_id rows: {before - len(selected_df)}")

base_rows = []
for _, row in selected_df.iterrows():
    raw = {col: clean_text(row[col]) for col in selected_df.columns}
    answer_raw = raw.get(ANSWER_TEXT_COL, "")
    base_rows.append({
        "benchmark_id": raw.get(ID_COL, ""),
        "source_dataset": "kqapro",
        "source_stage": "stage6_manual_translation",
        "question_en_original": raw.get(QUESTION_EN_COL, ""),

        # Fill this field manually in JSONL if you prefer editing JSONL directly.
        "question_ru_manual": "",

        # This field is filled automatically from question_ru_manual in the final export step.
        "question_ru": "",

        "translation_status": "needs_manual_translation",
        "answer_text_original": answer_raw,
        "answer_text_items": parse_answer_text(answer_raw),
        "primary_reasoning_family": raw.get("primary_reasoning_family", ""),
        "program_signature": raw.get("program_signature", ""),
        "program_length": raw.get("program_length", ""),
        "hop_count_estimate": raw.get("hop_count_estimate", ""),
        "target_domain_ru": raw.get("target_domain_ru", ""),
        "manual_bucket": raw.get("manual_bucket", ""),
        "manual_decision": raw.get("manual_decision", ""),
        "russian_reconstruction_idea": raw.get("russian_reconstruction_idea", ""),
        "notes": raw.get("notes", ""),
        "metadata": raw,
    })

print(f"Prepared records: {len(base_rows)}")
base_rows[0] if base_rows else None

Selected rows: 72
Removed duplicate benchmark_id rows: 2
Prepared records: 72


{'benchmark_id': 'kqapro_train_013987',
 'source_dataset': 'kqapro',
 'source_stage': 'stage6_manual_translation',
 'question_en_original': 'Which big city was, in 1958, a twinned administrative body of Hamburg, and is the place of the story in Catch Me In You Can (the one that was nominated for Broadcast Film Critics Association Award for Best Film)?',
 'question_ru_manual': '',
 'question_ru': '',
 'translation_status': 'needs_manual_translation',
 'answer_text_original': 'Marseille',
 'answer_text_items': ['Marseille'],
 'primary_reasoning_family': 'multi_hop',
 'program_signature': 'Find -> Relate -> QFilterYear -> FilterConcept -> Find -> Relate -> Find -> And -> Relate -> FilterConcept -> And -> What',
 'program_length': '12',
 'hop_count_estimate': '3',
 'target_domain_ru': '',
 'manual_bucket': '',
 'manual_decision': '',
 'russian_reconstruction_idea': '',
 'notes': '',
 'metadata': {'benchmark_id': 'kqapro_train_013987',
  'primary_reasoning_family': 'multi_hop',
  'recommend

## 5. Build manual translation review table

In [41]:
existing_manual = load_existing_manual_translations()

review_rows = []
for rec in base_rows:
    saved = existing_manual.get(rec["benchmark_id"], {})
    review_rows.append({
        ID_COL: rec["benchmark_id"],
        "primary_reasoning_family": rec.get("primary_reasoning_family", ""),
        "program_length": rec.get("program_length", ""),
        "hop_count_estimate": rec.get("hop_count_estimate", ""),
        QUESTION_EN_COL: rec["question_en_original"],

        # Paste the Russian translation here. Existing values are preserved by benchmark_id.
        QUESTION_RU_MANUAL_COL: saved.get(QUESTION_RU_MANUAL_COL, ""),

        # Optional note for rows that need extra manual checking.
        TRANSLATION_COMMENT_COL: saved.get(TRANSLATION_COMMENT_COL, ""),

        ANSWER_TEXT_COL: rec.get("answer_text_original", ""),
        "target_domain_ru": rec.get("target_domain_ru", ""),
        "manual_bucket": rec.get("manual_bucket", ""),
        "russian_reconstruction_idea": rec.get("russian_reconstruction_idea", ""),
        "notes": rec.get("notes", ""),
    })

review_df = pd.DataFrame(review_rows)
review_csv_path = write_csv(REVIEW_CSV_OUT, review_df, overwrite=OVERWRITE_OUTPUT)
print(f"Manual translation review CSV written: {review_csv_path}")
print(f"Filled translations in written review CSV: {int(review_df[QUESTION_RU_MANUAL_COL].map(clean_text).str.len().gt(0).sum())}")
review_df.head(10)


Manual translation review CSV written: artifacts_kqapro_stage6/csv/kqapro_manual_translation_review_sheet.csv
Filled translations in written review CSV: 0


,benchmark_id,primary_reasoning_family,program_length,hop_count_estimate,question_en_original,question_ru_manual,translation_comment,answer_text,target_domain_ru,manual_bucket,russian_reconstruction_idea,notes
0,kqapro_train_013987,multi_hop,12,3,"Which big city was, in 1958, a twinned adminis...",,,Marseille,,,,
1,kqapro_train_035873,multi_hop,12,3,"Who is a cast member of ""Taken"" (which also ha...",,,Michael Jeter,,,,
2,kqapro_train_075279,multi_hop,10,3,"What person is a member, starting in 2012, of ...",,,Thiago Silva,,,,
3,kqapro_train_075408,multi_hop,12,2,Which Japanese prefecture consists of more tha...,,,Ōsaka Prefecture,,,,
4,kqapro_train_076295,multi_hop,12,2,What New York county in New York (the one that...,,,Greene County,,,,
5,kqapro_train_035011,multi_hop,12,2,Which visual artwork has its narrative locatio...,,,Face/Off,,,,
6,kqapro_train_085057,multi_hop,12,2,"Which television series contains ""Six Feet Und...",,,Everybody Loves Raymond,,,,
7,kqapro_train_090128,multi_hop,10,2,What person is the winner of the Screen Actors...,,,Jack Huston,,,,
8,kqapro_val_007523,multi_hop,9,2,Which person received the Cannes Film Festival...,,,Christoph Waltz,,,,
9,kqapro_train_080370,multi_hop,9,2,What is the beginning date that Viacom (that i...,,,1986,,,,


## 6. Export JSONL with manual translation slots

In [42]:
# Draft JSONL contains empty question_ru_manual/question_ru fields.
# You can either edit this JSONL directly or fill the review CSV and run the next section.
jsonl_draft_path = write_jsonl(JSONL_DRAFT_OUT, base_rows, overwrite=OVERWRITE_OUTPUT)
print(f"Draft JSONL written: {jsonl_draft_path}")
print(f"Rows written: {len(base_rows)}")

Draft JSONL written: artifacts_kqapro_stage6/jsonl/kqapro_selected_manual_translation_draft.jsonl
Rows written: 72


## 7. Optional: rebuild final JSONL from edited review CSV

In [43]:
# After filling the review CSV manually, run this cell.
# The cell automatically picks the review CSV with the largest number of filled translations.
RUN_FINAL_EXPORT = True

if RUN_FINAL_EXPORT:
    review_csv_for_final = find_best_review_csv()
    assert review_csv_for_final is not None, f"No review CSV found under: {CSV_DIR.resolve()}"

    edited_review_df = pd.read_csv(review_csv_for_final, dtype=str, keep_default_na=False)
    assert ID_COL in edited_review_df.columns, f"Missing column in review CSV: {ID_COL}"
    assert QUESTION_RU_MANUAL_COL in edited_review_df.columns, f"Missing column in review CSV: {QUESTION_RU_MANUAL_COL}"

    ru_by_id = {
        clean_text(row.get(ID_COL, "")): clean_text(row.get(QUESTION_RU_MANUAL_COL, ""))
        for _, row in edited_review_df.iterrows()
    }
    comment_by_id = {
        clean_text(row.get(ID_COL, "")): clean_text(row.get(TRANSLATION_COMMENT_COL, ""))
        for _, row in edited_review_df.iterrows()
    }

    final_rows = []
    for rec in base_rows:
        out = dict(rec)
        qru = ru_by_id.get(rec["benchmark_id"], "")
        out["question_ru_manual"] = qru
        out["question_ru"] = qru
        out["translation_comment"] = comment_by_id.get(rec["benchmark_id"], "")
        out["translation_status"] = "manual_translated" if qru else "needs_manual_translation"
        final_rows.append(out)

    translated_count = sum(1 for row in final_rows if clean_text(row.get("question_ru", "")))
    missing_count = len(final_rows) - translated_count

    jsonl_final_path = write_jsonl(JSONL_FINAL_OUT, final_rows, overwrite=OVERWRITE_OUTPUT)
    print(f"Review CSV used for final export: {review_csv_for_final}")
    print(f"Final JSONL written: {jsonl_final_path}")
    print(f"Rows written: {len(final_rows)}")
    print(f"Translated rows: {translated_count}")
    print(f"Missing translations: {missing_count}")

    if missing_count > 0:
        missing_ids = [row["benchmark_id"] for row in final_rows if not clean_text(row.get("question_ru", ""))]
        print("First missing benchmark_id values:", missing_ids[:10])
else:
    print("Final export skipped. Fill the review CSV, then set RUN_FINAL_EXPORT = True.")


Review CSV used for final export: artifacts_kqapro_stage6/csv/kqapro_manual_translation_review_sheet.csv
Final JSONL written: artifacts_kqapro_stage6/jsonl/kqapro_selected_ru_final.jsonl
Rows written: 72
Translated rows: 72
Missing translations: 0


## 8. Quick checks

In [44]:
# Validate the final rows if the final export cell has been run; otherwise validate the draft rows.
check_rows = final_rows if "final_rows" in globals() else base_rows
check_df = pd.DataFrame(check_rows)

checks = {
    "rows": len(check_df),
    "unique_benchmark_ids": check_df["benchmark_id"].nunique() if len(check_df) else 0,
    "empty_question_en_original": int((check_df["question_en_original"].map(clean_text).str.len() == 0).sum()) if len(check_df) else 0,
    "empty_question_ru_manual": int((check_df["question_ru_manual"].map(clean_text).str.len() == 0).sum()) if len(check_df) else 0,
}

print(json.dumps(checks, ensure_ascii=False, indent=2))

sample_cols = [
    "benchmark_id",
    "question_en_original",
    "question_ru_manual",
    "question_ru",
    "answer_text_original",
    "translation_status",
]
check_df[sample_cols].head(10)


{
  "rows": 72,
  "unique_benchmark_ids": 72,
  "empty_question_en_original": 0,
  "empty_question_ru_manual": 0
}


,benchmark_id,question_en_original,question_ru_manual,question_ru,answer_text_original,translation_status
0,kqapro_train_013987,"Which big city was, in 1958, a twinned adminis...",Какой крупный город в 1958 году стал городом-п...,Какой крупный город в 1958 году стал городом-п...,Marseille,manual_translated
1,kqapro_train_035873,"Who is a cast member of ""Taken"" (which also ha...","Кто из актёров фильма «Заложница», где также с...","Кто из актёров фильма «Заложница», где также с...",Michael Jeter,manual_translated
2,kqapro_train_075279,"What person is a member, starting in 2012, of ...",Какой человек с 2012 года является членом футб...,Какой человек с 2012 года является членом футб...,Thiago Silva,manual_translated
3,kqapro_train_075408,Which Japanese prefecture consists of more tha...,Какая японская префектура более чем на 15% сос...,Какая японская префектура более чем на 15% сос...,Ōsaka Prefecture,manual_translated
4,kqapro_train_076295,What New York county in New York (the one that...,"В каком округе штата Нью-Йорк, где был выпущен...","В каком округе штата Нью-Йорк, где был выпущен...",Greene County,manual_translated
5,kqapro_train_035011,Which visual artwork has its narrative locatio...,"Какое визуальное произведение, действие которо...","Какое визуальное произведение, действие которо...",Face/Off,manual_translated
6,kqapro_train_085057,"Which television series contains ""Six Feet Und...",Какой телесериал включает эпизод «Six Feet Und...,Какой телесериал включает эпизод «Six Feet Und...,Everybody Loves Raymond,manual_translated
7,kqapro_train_090128,What person is the winner of the Screen Actors...,Какой человек получил премию Гильдии киноактёр...,Какой человек получил премию Гильдии киноактёр...,Jack Huston,manual_translated
8,kqapro_val_007523,Which person received the Cannes Film Festival...,Какой человек получил приз Каннского кинофести...,Какой человек получил приз Каннского кинофести...,Christoph Waltz,manual_translated
9,kqapro_train_080370,What is the beginning date that Viacom (that i...,"С какой даты Viacom, владеющая Paramount Pictu...","С какой даты Viacom, владеющая Paramount Pictu...",1986,manual_translated


In [45]:
report_path = safe_write_path(REPORT_OUT, overwrite=OVERWRITE_OUTPUT)
with report_path.open("w", encoding="utf-8") as f:
    f.write("# Stage 6 manual translation conversion report")
    f.write(f"- Created at: {datetime.now().isoformat(timespec='seconds')}")
    f.write(f"- Input CSV: `{INPUT_CSV}`")
    f.write(f"- Review CSV: `{globals().get('review_csv_path', '')}`")
    f.write(f"- Review CSV used for final export: `{globals().get('review_csv_for_final', '')}`")
    f.write(f"- Draft JSONL: `{globals().get('jsonl_draft_path', '')}`")
    f.write(f"- Final JSONL: `{globals().get('jsonl_final_path', '')}`")
    f.write(f"- Rows: {checks['rows']}")
    f.write(f"- Unique benchmark IDs: {checks['unique_benchmark_ids']}")
    f.write(f"- Empty English questions: {checks['empty_question_en_original']}")
    f.write(f"- Empty manual Russian translations: {checks['empty_question_ru_manual']}")

print(f"Report written: {report_path}")


Report written: artifacts_kqapro_stage6/reports/stage6_manual_translation_conversion_report.md


## 9. Notes and conclusions

The notebook creates two main artifacts:

- `artifacts_kqapro_stage6/csv/kqapro_manual_translation_review_sheet.csv` — the easiest file for manual translation. Fill `question_ru_manual`.
- `artifacts_kqapro_stage6/jsonl/kqapro_selected_manual_translation_draft.jsonl` — JSONL with explicit empty slots: `question_ru_manual` and `question_ru`.

Recommended workflow:

1. Run the notebook once.
2. Open the review CSV and paste Russian translations into `question_ru_manual`.
3. Return to section 7, set `RUN_FINAL_EXPORT = True`, and rerun the cell.
4. Use `artifacts_kqapro_stage6/jsonl/kqapro_selected_ru_final.jsonl` as the final translated JSONL.

The English original, raw answers, KQA Pro program signature, and all source metadata remain in every JSONL row, so each translated query stays auditable.